### Ingestion del archivo "genre.csv"

In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
%fs ls abfss://bronze@moviehistory310785.dfs.core.windows.net/

path,name,size,modificationTime
abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-16/,2024-12-16/,0,0
abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-23/,2024-12-23/,0,0
abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-30/,2024-12-30/,0,0


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
genre_schema = StructType(fields= [
    StructField("genreId", IntegerType(), False),
    StructField("genreName", StringType(), True)
])

In [0]:
genre_df = spark.read \
           .option("header", True) \
           .schema(genre_schema) \
           .csv(f"{bronze_folder_path}/{v_file_date}/genre.csv")

In [0]:
from pyspark.sql.functions import col

In [0]:
genre_selected_df = genre_df.select(col("genreId"), col("genreName"))

In [0]:
genre_renamed_df = genre_selected_df \
                   .withColumnRenamed("genreId", "genre_Id") \
                   .withColumnRenamed("genreName", "genre_Name")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
genre_final_df = add_ingestion_date(genre_renamed_df) \
                 .withColumn("environment", lit(v_environment)) \
                 .withColumn("file_date", lit(v_file_date))

In [0]:
display(genre_final_df)

genre_Id,genre_Name,ingestion_date,environment,file_date
12,Adventure,2026-09-10T20:07:37.2072Z,production,2024-12-30
14,Fantasy,2026-09-10T20:07:37.2072Z,production,2024-12-30
16,Animation,2026-09-10T20:07:37.2072Z,production,2024-12-30
18,Drama,2026-09-10T20:07:37.2072Z,production,2024-12-30
27,Horror,2026-09-10T20:07:37.2072Z,production,2024-12-30
28,Action,2026-09-10T20:07:37.2072Z,production,2024-12-30
35,Comedy,2026-09-10T20:07:37.2072Z,production,2024-12-30
36,History,2026-09-10T20:07:37.2072Z,production,2024-12-30
37,Western,2026-09-10T20:07:37.2072Z,production,2024-12-30
53,Thriller,2026-09-10T20:07:37.2072Z,production,2024-12-30


In [0]:
genre_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genres")

In [0]:
display(spark.read.table("movie_silver.genres"))

genre_Id,genre_Name,ingestion_date,environment,file_date
12,Adventure,2026-09-10T20:07:38.225186Z,production,2024-12-30
14,Fantasy,2026-09-10T20:07:38.225186Z,production,2024-12-30
16,Animation,2026-09-10T20:07:38.225186Z,production,2024-12-30
18,Drama,2026-09-10T20:07:38.225186Z,production,2024-12-30
27,Horror,2026-09-10T20:07:38.225186Z,production,2024-12-30
28,Action,2026-09-10T20:07:38.225186Z,production,2024-12-30
35,Comedy,2026-09-10T20:07:38.225186Z,production,2024-12-30
36,History,2026-09-10T20:07:38.225186Z,production,2024-12-30
37,Western,2026-09-10T20:07:38.225186Z,production,2024-12-30
53,Thriller,2026-09-10T20:07:38.225186Z,production,2024-12-30


In [0]:
%sql
SELECT * FROM movie_silver.genres

genre_Id,genre_Name,ingestion_date,environment,file_date
12,Adventure,2026-09-10T20:07:38.225186Z,production,2024-12-30
14,Fantasy,2026-09-10T20:07:38.225186Z,production,2024-12-30
16,Animation,2026-09-10T20:07:38.225186Z,production,2024-12-30
18,Drama,2026-09-10T20:07:38.225186Z,production,2024-12-30
27,Horror,2026-09-10T20:07:38.225186Z,production,2024-12-30
28,Action,2026-09-10T20:07:38.225186Z,production,2024-12-30
35,Comedy,2026-09-10T20:07:38.225186Z,production,2024-12-30
36,History,2026-09-10T20:07:38.225186Z,production,2024-12-30
37,Western,2026-09-10T20:07:38.225186Z,production,2024-12-30
53,Thriller,2026-09-10T20:07:38.225186Z,production,2024-12-30


In [0]:
dbutils.notebook.exit("Success")